# Taller Interactivo: Ley de Gauss y Aplicaciones en Ingeniería
### Escuela Colombiana de Ingeniería Julio Garavito — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
**Versión blindada para Google Colab.** Cada punto es una celda **autónoma**: puedes
ejecutar cualquiera sin depender de las demás. Los sliders y el quiz vienen incluidos.

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Si un slider no aparece, vuelve a
> ejecutar esa celda (los widgets de Colab ya quedan habilitados automáticamente).


## 🔧 Celda 0 — Comprobación rápida (opcional)

In [ ]:
# Ejecuta esto si quieres asegurarte de que Colab tiene todo.
# En Colab normalmente NO hace falta instalar nada.
# !pip install ipywidgets -q
import numpy, scipy, matplotlib, ipywidgets
print("NumPy", numpy.__version__, "| SciPy", scipy.__version__,
      "| Matplotlib", matplotlib.__version__, "| ipywidgets", ipywidgets.__version__)
print("Todo listo \u2714  — cada celda de punto es autónoma.")

---
## Punto 1 — Sensor esférico concéntrico (Ing. Electrónica/Mecánica)
Núcleo conductor $a=2{,}00$ cm, $Q_1=+8{,}00$ nC; cáscara conductora $b=5{,}00$, $c=7{,}00$ cm, $Q_2=-3{,}00$ nC.
$$E(r)=\frac{Q_{enc}(r)}{4\pi\epsilon_0 r^2}$$

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


def punto1(Q1_nC=8.0, Q2_nC=-3.0, r_cm=10.0):
    a, b, c = 0.02, 0.05, 0.07
    Q1, Q2 = Q1_nC*1e-9, Q2_nC*1e-9
    r = np.linspace(0.001, 0.15, 800); E = np.zeros_like(r)
    for i, rr in enumerate(r):
        if rr < a: q=0.0
        elif rr < b: q=Q1
        elif rr < c: q=0.0
        else: q=Q1+Q2
        E[i]=k*q/rr**2
    rp=r_cm/100
    q = 0.0 if rp<a else Q1 if rp<b else 0.0 if rp<c else Q1+Q2
    Ep=k*q/rp**2; Phi=q/eps0
    plt.figure(figsize=(9,5)); plt.plot(r*100,E,lw=2,color='navy')
    plt.axvspan(0,a*100,alpha=.15,color='gray',label='núcleo conductor')
    plt.axvspan(b*100,c*100,alpha=.15,color='orange',label='cáscara conductora')
    plt.scatter([r_cm],[Ep],color='red',s=60,zorder=5,label=f'r={r_cm:.1f} cm')
    plt.xlabel('r (cm)'); plt.ylabel('E(r) (V/m)')
    plt.title('Campo radial — sensor esférico'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f"En r={r_cm:.2f} cm:  Q_enc={q*1e9:+.2f} nC")
    print(f"  Flujo  Φ_E = {Phi:,.3f} N·m²/C")
    print(f"  Campo  E   = {Ep:,.1f} V/m  ≈ {Ep:.3g} V/m")

interact(punto1,
    Q1_nC=FloatSlider(value=8.0,min=-10,max=10,step=0.5,description='Q₁ (nC)'),
    Q2_nC=FloatSlider(value=-3.0,min=-10,max=10,step=0.5,description='Q₂ (nC)'),
    r_cm =FloatSlider(value=10.0,min=0.5,max=14,step=0.5,description='r (cm)'));

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("a) Flujo neto Φ_E en r=10,0 cm:", 564.972, 0.03, "N·m²/C",
              "Φ=(Q1+Q2)/ε₀=(5,00 nC)/ε₀=565 N·m²/C.")
quiz_numerico("a) Campo E en r=10,0 cm:", 4495.9, 0.03, "V/m",
              "E=k(Q1+Q2)/r²=4,50×10³ V/m.")
quiz_opcion_multiple("b) Cáscara a tierra: flujo en r=3,00 cm (a<r<b):",
    ["Aumenta","Permanece constante: sólo depende de Q1 encerrada","Se vuelve cero","Cambia de signo"],
    1,"En a<r<b la carga encerrada es Q1; la tierra sólo afecta la carga externa.")


---
## Punto 2 — Ducto cilíndrico en campo uniforme (Ing. Civil/Ambiental)
$R=0{,}500$ m, $L=3{,}00$ m, $\vec E=(4{,}00\times10^3)\hat i+(3{,}00\times10^3)\hat k$ V/m.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


def punto2(Ex_k=4.0, Ez_k=3.0, R=0.5, L=3.0):
    Ex, Ez = Ex_k*1e3, Ez_k*1e3
    A=np.pi*R**2
    phi_sup=Ez*A; phi_inf=-Ez*A; phi_lat=0.0
    phi_net=phi_sup+phi_inf+phi_lat
    fig=plt.figure(figsize=(7,6)); ax=fig.add_subplot(111,projection='3d')
    z=np.linspace(-L/2,L/2,30); th=np.linspace(0,2*np.pi,40); Z,T=np.meshgrid(z,th)
    ax.plot_surface(R*np.cos(T),R*np.sin(T),Z,alpha=.25,color='steelblue')
    ax.quiver(0,0,0,Ex/2000,0,Ez/2000,color='red',lw=2,arrow_length_ratio=.15)
    ax.set_title('Ducto en campo uniforme'); ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    plt.tight_layout(); plt.show()
    print(f"Φ tapa superior = {phi_sup:,.2f} N·m²/C")
    print(f"Φ tapa inferior = {phi_inf:,.2f} N·m²/C")
    print(f"Φ lateral (x, simetría) = {phi_lat:,.2f} N·m²/C")
    print(f"Φ NETO = {phi_net:,.2f} N·m²/C → sin carga encerrada")

interact(punto2,
    Ex_k=FloatSlider(value=4.0,min=0,max=10,step=0.5,description='Eₓ (kV/m)'),
    Ez_k=FloatSlider(value=3.0,min=0,max=10,step=0.5,description='E_z (kV/m)'),
    R=FloatSlider(value=0.5,min=0.2,max=1.0,step=0.1,description='R (m)'),
    L=FloatSlider(value=3.0,min=1,max=5,step=0.5,description='L (m)'));

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("a) Flujo por la tapa superior (Ez=3,00 kV/m, R=0,500 m):",
    2356.19,0.03,"N·m²/C","Φ=Ez·πR²=3000·π·0,25=2,36×10³ N·m²/C.")
quiz_opcion_multiple("a) Flujo neto total sobre la superficie cerrada:",
    ["Ez·πR²","Cero","(Ex+Ez)·πR²","Depende de L"],1,
    "Sin carga encerrada Φ_neto=0.")
quiz_opcion_multiple("b) Diferencia entre Φ=0 y E=0:",
    ["Son equivalentes",
     "Φ=0 = no hay carga neta encerrada; E=0 sería campo nulo en cada punto (aquí falso)",
     "E=0 implica Φ=0 pero no al revés","Ninguna relación"],1,
    "Φ=0 es balance global de líneas; el campo NO es cero en la superficie.")


---
## Punto 3 — Línea de carga y distancia RETIE (Ing. Eléctrica/Civil)
$\lambda=+15{,}0\ \mu$C/m, $E_{rup}=3{,}00\times10^6$ V/m.
$$r_{crítico}=\frac{\lambda}{2\pi\epsilon_0 E_{rup}}$$

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


def punto3(lam_uC=15.0, Erup_M=3.0):
    lam=lam_uC*1e-6; Erup=Erup_M*1e6
    r=np.linspace(0.01,0.5,800); E=lam/(2*np.pi*eps0*r)
    rc=lam/(2*np.pi*eps0*Erup)
    plt.figure(figsize=(9,5)); plt.plot(r*100,E,lw=2,color='darkgreen')
    plt.axhline(Erup,color='red',ls='--',label=f'E ruptura={Erup:.2g} V/m')
    plt.axvline(rc*100,color='purple',ls=':',label=f'r_crítico={rc*100:.2f} cm')
    plt.yscale('log'); plt.xlabel('r (cm)'); plt.ylabel('E(r) (V/m, log)')
    plt.title('Línea de carga vs distancia'); plt.legend(); plt.grid(alpha=.3,which='both')
    plt.tight_layout(); plt.show()
    print(f"r_crítico = {rc:.4f} m = {rc*100:.3f} cm")

interact(punto3,
    lam_uC=FloatSlider(value=15.0,min=1,max=40,step=1,description='λ (µC/m)'),
    Erup_M=FloatSlider(value=3.0,min=1,max=5,step=0.5,description='E_rup (MV/m)'));

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("a) Distancia crítica r_crítico (λ=15,0 µC/m):",8.992,0.03,"cm",
    "r=λ/(2πε₀·E_rup)=0,0899 m=8,99 cm.")
quiz_opcion_multiple("b) Si λ→2λ, r_crítico:",
    ["Se reduce a la mitad","Se duplica (r∝λ)","Se cuadruplica","No cambia"],1,
    "r_crítico∝λ: al duplicar λ, se duplica la distancia.")


---
## Punto 4 — Cable no conductor + tubo conductor (Ing. Industrial/Mecánica)
Cable $a=1{,}00$ cm, $\rho=+4{,}00\ \mu$C/m³; tubo conductor neutro $b=3{,}00$, $c=5{,}00$ cm.
Interior: $E=\rho r/(2\epsilon_0)$; dentro del conductor $E=0$.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


def punto4(rho_uC=4.0, r_cm=0.5):
    a,b,c=0.01,0.03,0.05; rho=rho_uC*1e-6; lam=rho*np.pi*a**2
    r=np.linspace(0.0005,0.08,900); E=np.zeros_like(r)
    for i,rr in enumerate(r):
        if rr<a: E[i]=rho*rr/(2*eps0)
        elif rr<b: E[i]=lam/(2*np.pi*eps0*rr)
        elif rr<c: E[i]=0.0
        else: E[i]=lam/(2*np.pi*eps0*rr)
    rp=r_cm/100
    if rp<a: Ep=rho*rp/(2*eps0)
    elif rp<b: Ep=lam/(2*np.pi*eps0*rp)
    elif rp<c: Ep=0.0
    else: Ep=lam/(2*np.pi*eps0*rp)
    plt.figure(figsize=(9,5)); plt.plot(r*100,E,lw=2,color='maroon')
    plt.axvspan(0,a*100,alpha=.15,color='gold',label='cable no conductor')
    plt.axvspan(b*100,c*100,alpha=.15,color='gray',label='tubo conductor')
    plt.scatter([r_cm],[Ep],color='blue',s=60,zorder=5,label=f'r={r_cm:.2f} cm')
    plt.xlabel('r (cm)'); plt.ylabel('E(r) (V/m)')
    plt.title('Cable dieléctrico + blindaje'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    sigb=-lam/(2*np.pi*b); sigc=lam/(2*np.pi*c)
    print(f"En r={r_cm:.2f} cm: E={Ep:.3g} V/m")
    print(f"λ_cable={lam:.3g} C/m ; σ_b={sigb:.3g} C/m² ; σ_c={sigc:.3g} C/m²")

interact(punto4,
    rho_uC=FloatSlider(value=4.0,min=1,max=10,step=0.5,description='ρ (µC/m³)'),
    r_cm =FloatSlider(value=0.5,min=0.1,max=7,step=0.1,description='r (cm)'));

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("a-i) Campo en r=0,500 cm:",1130.0,0.05,"V/m",
    "E=ρr/(2ε₀)≈1,13×10³ V/m.")
quiz_opcion_multiple("a-ii) Campo en r=4,00 cm (dentro del tubo):",
    ["Máximo","Cero (conductor en equilibrio)","Igual al del cable","Negativo"],1,
    "En un conductor en equilibrio E=0.")
quiz_numerico("b) σ_c (pared externa):",4.0e-9,0.06,"C/m²",
    "σ_c=+λ/(2πc)≈+4,0×10⁻⁹ C/m²; σ_b es negativa.")


---
## Punto 5 — Dos láminas planas infinitas (Ing. Civil/Sistemas)
$\sigma_1=+8{,}85$, $\sigma_2=-8{,}85$ nC/m², separadas 10 cm. Una lámina: $E=\sigma/(2\epsilon_0)$.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


def punto5(sig1_n=8.85, sig2_n=-8.85):
    sig1,sig2=sig1_n*1e-9,sig2_n*1e-9; x1,x2=0.0,0.10
    x=np.linspace(-0.2,0.3,500)
    def campo(xx): return np.sign(xx-x1)*sig1/(2*eps0)+np.sign(xx-x2)*sig2/(2*eps0)
    E=np.array([campo(xx) for xx in x])
    plt.figure(figsize=(9,5)); plt.plot(x*100,E,lw=2,color='teal')
    plt.axvline(0,color='red',ls='--',label='σ₁'); plt.axvline(10,color='blue',ls='--',label='σ₂')
    plt.xlabel('x (cm)'); plt.ylabel('E_x neto (V/m)')
    plt.title('Superposición de dos láminas'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f"Una lámina: σ/2ε₀={sig1/(2*eps0):.3g} V/m")
    print(f"ENTRE={campo(0.05):.3g} V/m ; EXTERIOR={campo(0.25):.3g} V/m")

interact(punto5,
    sig1_n=FloatSlider(value=8.85,min=-15,max=15,step=0.85,description='σ₁ (nC/m²)'),
    sig2_n=FloatSlider(value=-8.85,min=-15,max=15,step=0.85,description='σ₂ (nC/m²)'));

print("\n--- Cuestionario Punto 5 ---")
quiz_numerico("a) Campo de UNA lámina (σ=8,85 nC/m²):",500.0,0.05,"V/m",
    "E=σ/(2ε₀)=500 V/m.")
quiz_numerico("a) Campo ENTRE las láminas (signos opuestos):",1000.0,0.05,"V/m",
    "Se suman: E=σ/ε₀=1,00×10³ V/m.")
quiz_opcion_multiple("b) Si σ₁=σ₂=+8,85 nC/m², campo intermedio:",
    ["Se duplica","Cero: se cancelan","500 V/m","1000 V/m"],1,
    "Con cargas iguales los campos se oponen entre las láminas → E=0.")


---
## Punto 6 — Esfera dieléctrica $\rho(r)=C\,r$ (Ing. Eléctrica/Mecánica)
$R=6{,}00$ cm, $C=2{,}00\ \mu$C/m⁴. $Q_{enc}(r)=\pi C r^4$, $E(r)=Cr^2/(4\epsilon_0)$.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np, scipy.constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass
eps0 = const.epsilon_0
k = 1/(4*np.pi*eps0)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            print("\u2705 \u00a1Correcto!" if radio.index==indice_correcto else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion=""):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info')
    salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try:
                v = float(caja.value.replace(",", "."))
            except ValueError:
                print("\u26a0 Escribe un n\u00famero (punto o coma decimal)."); return
            if abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel:
                print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else:
                print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
    boton.on_click(revisar)
    display(VBox([titulo, caja, boton, salida]))
# --- fin setup ---


from scipy.integrate import quad
def punto6(C_uC=2.0, R_cm=6.0, r_cm=6.0):
    C=C_uC*1e-6; R=R_cm/100
    r=np.linspace(0.001,0.12,800); E=np.zeros_like(r)
    for i,rr in enumerate(r):
        Q = np.pi*C*rr**4 if rr<=R else np.pi*C*R**4
        E[i]=k*Q/rr**2
    Qnum,_=quad(lambda rr:C*rr*4*np.pi*rr**2,0,R)
    rp=r_cm/100
    Qp = np.pi*C*min(rp,R)**4 if rp<=R else np.pi*C*R**4
    Ep=k*Qp/rp**2
    plt.figure(figsize=(9,5)); plt.plot(r*100,E,lw=2,color='indigo')
    plt.axvspan(0,R*100,alpha=.12,color='violet',label='esfera dieléctrica')
    plt.scatter([r_cm],[Ep],color='red',s=60,zorder=5,label=f'r={r_cm:.1f} cm')
    plt.xlabel('r (cm)'); plt.ylabel('E(r) (V/m)')
    plt.title('Esfera con ρ(r)=C·r'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f"Q_total (πCR⁴)={np.pi*C*R**4:.3g} C ; (scipy)={Qnum:.3g} C")
    print(f"En r={r_cm:.1f} cm: E={Ep:.3g} V/m")

interact(punto6,
    C_uC=FloatSlider(value=2.0,min=0.5,max=5,step=0.5,description='C (µC/m⁴)'),
    R_cm=FloatSlider(value=6.0,min=3,max=10,step=0.5,description='R (cm)'),
    r_cm=FloatSlider(value=6.0,min=0.5,max=12,step=0.5,description='r (cm)'));

print("\n--- Cuestionario Punto 6 ---")
quiz_numerico("a) Carga total Q_total:",8.14e-11,0.05,"C",
    "Q=πCR⁴≈8,14×10⁻¹¹ C.")
quiz_numerico("a) Campo E en r=R=6,00 cm:",203.0,0.05,"V/m",
    "E=kQ/R²=CR²/(4ε₀)≈203 V/m.")
quiz_opcion_multiple("b) ¿Dónde crece más rápido E interior?",
    ["Densidad uniforme (E∝r)","Densidad ρ∝r (E∝r², más rápido)","Igual","Ninguno"],1,
    "ρ uniforme → E∝r; ρ∝r → E∝r².")


---
## ✅ Fin del taller
Cada celda es independiente. Si un slider no se muestra, re-ejecuta esa celda.
*Física de Electricidad y Magnetismo — 2026-2.*